In [11]:
# ============================================================
# ASTRA VITALIS
# Astronaut Health Intelligence Research Notebook
# SOLVYRIS | Space Health Research Division
# ============================================================

%pip install matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

# Reproducibility
np.random.seed(42)

# Project paths
PROJECT_DIR = Path.cwd() 
NASA_DATA_PATH = PROJECT_DIR / "nasa_research" / "OSD575_CMP.csv"

print("ASTRA VITALIS — RESEARCH ENVIRONMENT")
print("------------------------------------")
print("Research environment initialized.")

if NASA_DATA_PATH.exists():
    print("NASA dataset: FOUND")
else:
    print("NASA dataset: NOT FOUND")
    print("Check that the notebook is inside the Astra-Vitalis folder.")

Note: you may need to restart the kernel to use updated packages.
ASTRA VITALIS — RESEARCH ENVIRONMENT
------------------------------------
Research environment initialized.
NASA dataset: FOUND



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

print("All research libraries imported successfully!")

All research libraries imported successfully!


In [13]:
# ============================================================
# 01. DATASET LOADING
# ============================================================

from pathlib import Path

NASA_DATA_PATH = Path.cwd() / "nasa_research" / "OSD575_CMP.csv"

nasa = pd.read_csv(NASA_DATA_PATH)

print("NASA OSD-575 DATASET LOADED")
print("=" * 45)

print("Dataset shape:", nasa.shape)
print("Number of samples:", len(nasa))
print("Number of columns:", len(nasa.columns))

print("\nFirst five records:")
display(nasa.head()) 

NASA OSD-575 DATASET LOADED
Dataset shape: (28, 58)
Number of samples: 28
Number of columns: 58

First five records:


,Sample Name,albumin_value_gram_per_deciliter,albumin_range_min_gram_per_deciliter,albumin_range_max_gram_per_deciliter,albumin_to_globulin_ratio_value,albumin_to_globulin_ratio_range_min,albumin_to_globulin_ratio_range_max,alkaline_phosphatase_value_units_per_liter,alkaline_phosphatase_range_min_units_per_liter,alkaline_phosphatase_range_max_units_per_liter,...,potassium_range_max_millimol_per_liter,total_protein_value_gram_per_deciliter,total_protein_range_min_gram_per_deciliter,total_protein_range_max_gram_per_deciliter,sodium_value_millimol_per_liter,sodium_range_min_millimol_per_liter,sodium_range_max_millimol_per_liter,urea_nitrogen_bun_value_milligram_per_deciliter,urea_nitrogen_bun_range_min_milligram_per_deciliter,urea_nitrogen_bun_range_max_milligram_per_deciliter
0,C001_serum_L-3,4.9,3.6,5.1,2.0,1,2.5,45,36,130,...,5.3,7.4,6.1,8.1,142,135,146,20,7,25
1,C001_serum_L-44,5.0,3.6,5.1,1.9,1,2.5,46,36,130,...,5.3,7.7,6.1,8.1,139,135,146,19,7,25
2,C001_serum_L-92,5.0,3.6,5.1,2.2,1,2.5,51,36,130,...,5.3,7.3,6.1,8.1,141,135,146,25,7,25
3,C001_serum_R+1,4.9,3.6,5.1,1.9,1,2.5,52,36,130,...,5.3,7.5,6.1,8.1,138,135,146,18,7,25
4,C001_serum_R+194,4.6,3.6,5.1,1.9,1,2.5,50,36,130,...,5.3,7.0,6.1,8.1,137,135,146,28,7,25


In [14]:
# ============================================================
# 02. EXPLORATORY DATA ANALYSIS
# ============================================================

print("ASTRA VITALIS — DATA EXPLORATION")
print("=" * 50)

# Dataset information
print("\n1. DATASET STRUCTURE")
print("-" * 30)
nasa.info()

# Statistical summary
print("\n2. STATISTICAL SUMMARY")
print("-" * 30)
display(nasa.describe().T)

# Missing values
print("\n3. MISSING VALUE ANALYSIS")
print("-" * 30)

missing = nasa.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    display(missing.to_frame(name="Missing Values"))
else:
    print("No missing values found.")

# Participant identification
print("\n4. PARTICIPANT DISTRIBUTION")
print("-" * 30)

nasa["Participant"] = nasa["Sample Name"].str.extract(r"^(C\d+)")

display(nasa["Participant"].value_counts().sort_index())

# Timepoint identification
nasa["Timepoint"] = nasa["Sample Name"].str.extract(
    r"(L-\d+|R\+\d+)$"
)

print("\n5. MISSION TIMEPOINTS")
print("-" * 30)
display(nasa["Timepoint"].value_counts())

ASTRA VITALIS — DATA EXPLORATION

1. DATASET STRUCTURE
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 58 columns):
 #   Column                                                                            Non-Null Count  Dtype  
---  ------                                                                            --------------  -----  
 0   Sample Name                                                                       28 non-null     str    
 1   albumin_value_gram_per_deciliter                                                  28 non-null     float64
 2   albumin_range_min_gram_per_deciliter                                              28 non-null     float64
 3   albumin_range_max_gram_per_deciliter                                              28 non-null     float64
 4   albumin_to_globulin_ratio_value                                                   28 non-null     float64
 5   albumin_to_globulin_ratio_range_min     

,count,mean,std,min,25%,50%,75%,max
albumin_value_gram_per_deciliter,28.0,4.589286,2.726268e-01,4.20,4.4000,4.55,4.9000,5.10
albumin_range_min_gram_per_deciliter,28.0,3.600000,9.044766e-16,3.60,3.6000,3.60,3.6000,3.60
albumin_range_max_gram_per_deciliter,28.0,5.100000,0.000000e+00,5.10,5.1000,5.10,5.1000,5.10
albumin_to_globulin_ratio_value,28.0,1.664286,2.180966e-01,1.30,1.5000,1.60,1.8250,2.20
albumin_to_globulin_ratio_range_min,28.0,1.000000,0.000000e+00,1.00,1.0000,1.00,1.0000,1.00
albumin_to_globulin_ratio_range_max,28.0,2.500000,0.000000e+00,2.50,2.5000,2.50,2.5000,2.50
alkaline_phosphatase_value_units_per_liter,28.0,54.178571,1.163346e+01,31.00,46.0000,53.00,62.7500,78.00
alkaline_phosphatase_range_min_units_per_liter,28.0,35.535714,1.933757e+00,31.00,36.0000,36.00,36.2500,37.00
alkaline_phosphatase_range_max_units_per_liter,28.0,135.035714,1.070300e+01,125.00,130.0000,130.00,135.7500,153.00
alt_value_units_per_liter,28.0,20.000000,9.936838e+00,9.00,13.0000,16.00,23.2500,43.00



3. MISSING VALUE ANALYSIS
------------------------------


,Missing Values
egfr_african_american_range_max_milliliter_per_minute_per_1.73_meter_squared,28
egfr_non_african_american_range_max_milliliter_per_minute_per_1.73_meter_squared,28
bun_to_creatinine_ratio_value,22
total_bilirubin_value_milligram_per_deciliter,1
bun_to_creatinine_ratio_range_max,1
bun_to_creatinine_ratio_range_min,1



4. PARTICIPANT DISTRIBUTION
------------------------------


Participant
C001    7
C002    7
C003    7
C004    7
Name: count, dtype: int64


5. MISSION TIMEPOINTS
------------------------------


Timepoint
L-3      4
L-44     4
L-92     4
R+1      4
R+194    4
R+45     4
R+82     4
Name: count, dtype: int64

In [15]:
print("ASTRA VITALIS — DATASET SUMMARY")
print("=" * 45)

print("Total samples:", len(nasa))
print("Total columns:", len(nasa.columns))

print("\nPARTICIPANTS")
display(nasa["Participant"].value_counts().sort_index())

print("\nMISSION TIMEPOINTS")
display(nasa["Timepoint"].value_counts())

print("\nMISSING VALUES")
missing = nasa.isnull().sum()
display(missing[missing > 0].to_frame("Missing Count"))

print("\nUNIQUE PARTICIPANTS:",
      nasa["Participant"].nunique())

print("UNIQUE TIMEPOINTS:",
      nasa["Timepoint"].nunique())

ASTRA VITALIS — DATASET SUMMARY
Total samples: 28
Total columns: 60

PARTICIPANTS


Participant
C001    7
C002    7
C003    7
C004    7
Name: count, dtype: int64


MISSION TIMEPOINTS


Timepoint
L-3      4
L-44     4
L-92     4
R+1      4
R+194    4
R+45     4
R+82     4
Name: count, dtype: int64


MISSING VALUES


,Missing Count
total_bilirubin_value_milligram_per_deciliter,1
bun_to_creatinine_ratio_value,22
bun_to_creatinine_ratio_range_min,1
bun_to_creatinine_ratio_range_max,1
egfr_african_american_range_max_milliliter_per_minute_per_1.73_meter_squared,28
egfr_non_african_american_range_max_milliliter_per_minute_per_1.73_meter_squared,28



UNIQUE PARTICIPANTS: 4
UNIQUE TIMEPOINTS: 7


In [16]:
# ============================================================
# 03. GLUCOSE ANALYSIS
# ============================================================

timepoint_order = [
    "L-92", "L-44", "L-3",
    "R+1", "R+45", "R+82", "R+194"
]

nasa["Timepoint"] = pd.Categorical(
    nasa["Timepoint"],
    categories=timepoint_order,
    ordered=True
)

glucose_column = "glucose_value_milligram_per_deciliter"

glucose_data = nasa.sort_values("Timepoint")

fig = px.line(
    glucose_data,
    x="Timepoint",
    y=glucose_column,
    color="Participant",
    markers=True,
    title="Inspiration4 — Glucose Measurements Across Mission Timepoints",
    labels={
        "Timepoint": "Mission Timepoint",
        glucose_column: "Glucose (mg/dL)",
        "Participant": "Participant"
    }
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Mission Timepoint",
    yaxis_title="Glucose (mg/dL)",
    legend_title="Participant"
)

fig.show()

In [17]:
# ============================================================
# 04. BLOOD CHEMISTRY REFERENCE-RANGE ANALYSIS
# ============================================================

parameters = {
    "Glucose": (
        "glucose_value_milligram_per_deciliter",
        "glucose_range_min_milligram_per_deciliter",
        "glucose_range_max_milligram_per_deciliter"
    ),
    "Albumin": (
        "albumin_value_gram_per_deciliter",
        "albumin_range_min_gram_per_deciliter",
        "albumin_range_max_gram_per_deciliter"
    ),
    "Creatinine": (
        "creatinine_value_milligram_per_deciliter",
        "creatinine_range_min_milligram_per_deciliter",
        "creatinine_range_max_milligram_per_deciliter"
    ),
    "Sodium": (
        "sodium_value_millimol_per_liter",
        "sodium_range_min_millimol_per_liter",
        "sodium_range_max_millimol_per_liter"
    )
}

results = []

for parameter, columns in parameters.items():

    value_col, min_col, max_col = columns

    for _, row in nasa.iterrows():

        value = row[value_col]
        lower = row[min_col]
        upper = row[max_col]

        if pd.isna(value) or pd.isna(lower) or pd.isna(upper):
            status = "Missing"
        elif value < lower:
            status = "Below"
        elif value > upper:
            status = "Above"
        else:
            status = "Within"

        results.append({
            "Participant": row["Participant"],
            "Timepoint": row["Timepoint"],
            "Parameter": parameter,
            "Value": value,
            "Reference Minimum": lower,
            "Reference Maximum": upper,
            "Status": status
        })

reference_analysis = pd.DataFrame(results)

print("BLOOD CHEMISTRY REFERENCE-RANGE ANALYSIS")
print("=" * 55)

summary = pd.crosstab(
    reference_analysis["Parameter"],
    reference_analysis["Status"]
)

display(summary)

print("\nOVERALL COMPARISON SUMMARY")

overall = reference_analysis["Status"].value_counts()

display(overall.to_frame("Observations"))

print("\nTOTAL COMPARISONS:", len(reference_analysis))

BLOOD CHEMISTRY REFERENCE-RANGE ANALYSIS


Status,Above,Below,Within
Parameter,,,
Albumin,0,0,28
Creatinine,1,0,27
Glucose,2,5,21
Sodium,0,0,28



OVERALL COMPARISON SUMMARY


,Observations
Status,
Within,104
Below,5
Above,3



TOTAL COMPARISONS: 112


In [18]:
# ============================================================
# 05. REFERENCE-RANGE FINDINGS VISUALIZATION
# ============================================================

status_summary = (
    reference_analysis
    .groupby(["Parameter", "Status"])
    .size()
    .reset_index(name="Count")
)

fig = px.bar(
    status_summary,
    x="Parameter",
    y="Count",
    color="Status",
    barmode="group",
    text="Count",
    title="NASA Inspiration4 — Blood Chemistry Reference-Range Findings",
    category_orders={
        "Parameter": ["Glucose", "Albumin", "Creatinine", "Sodium"],
        "Status": ["Below", "Within", "Above"]
    }
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Blood Chemistry Parameter",
    yaxis_title="Number of Observations",
    legend_title="Reference Range Status"
)

fig.show()

In [19]:
# ============================================================
# 06. OUT-OF-RANGE OBSERVATION INVESTIGATION
# ============================================================

out_of_range = reference_analysis[
    reference_analysis["Status"].isin(["Below", "Above"])
].copy()

out_of_range = out_of_range.sort_values(
    ["Parameter", "Participant", "Timepoint"]
)

print("OUT-OF-RANGE OBSERVATIONS")
print("=" * 55)

display(out_of_range)

print("\nTOTAL FLAGGED OBSERVATIONS:", len(out_of_range))

print("\nPARAMETER-WISE FLAG COUNT")

display(
    out_of_range.groupby(["Parameter", "Status"])
    .size()
    .reset_index(name="Count")
)

OUT-OF-RANGE OBSERVATIONS


,Participant,Timepoint,Parameter,Value,Reference Minimum,Reference Maximum,Status
56,C001,L-3,Creatinine,1.37,0.6,1.35,Above
12,C002,R+45,Glucose,44.00,65.0,99.00,Below
15,C003,L-44,Glucose,62.00,65.0,99.00,Below
17,C003,R+1,Glucose,103.00,65.0,99.00,Above
19,C003,R+45,Glucose,36.00,65.0,99.00,Below
21,C004,L-3,Glucose,56.00,65.0,99.00,Below
23,C004,L-92,Glucose,100.00,65.0,99.00,Above
26,C004,R+45,Glucose,43.00,65.0,99.00,Below



TOTAL FLAGGED OBSERVATIONS: 8

PARAMETER-WISE FLAG COUNT


,Parameter,Status,Count
0,Creatinine,Above,1
1,Glucose,Above,2
2,Glucose,Below,5


In [20]:
# ============================================================
# 07. MISSING DATA INVESTIGATION
# ============================================================

print("ASTRA VITALIS — DATA QUALITY ANALYSIS")
print("=" * 50)

# Missing values in original NASA columns
original_columns = [
    col for col in nasa.columns
    if col not in ["Participant", "Timepoint"]
]

missing_report = pd.DataFrame({
    "Missing Count": nasa[original_columns].isnull().sum(),
    "Missing Percentage": (
        nasa[original_columns].isnull().mean() * 100
    )
})

missing_report = missing_report[
    missing_report["Missing Count"] > 0
].sort_values("Missing Count", ascending=False)

print("\nCOLUMNS WITH MISSING VALUES")
display(missing_report)

print("\nTOTAL MISSING CELLS:",
      nasa[original_columns].isnull().sum().sum())

print("\nTOTAL DATA CELLS:",
      nasa[original_columns].size)

# Visualise missing values
if not missing_report.empty:

    fig = px.bar(
        missing_report.reset_index(),
        x="index",
        y="Missing Count",
        title="NASA OSD-575 — Missing Data Distribution",
        labels={
            "index": "Dataset Column",
            "Missing Count": "Missing Observations"
        }
    )

    fig.update_layout(
        template="plotly_dark",
        xaxis_tickangle=-45
    )

    fig.show()

else:
    print("No missing values found.")

ASTRA VITALIS — DATA QUALITY ANALYSIS

COLUMNS WITH MISSING VALUES


,Missing Count,Missing Percentage
egfr_african_american_range_max_milliliter_per_minute_per_1.73_meter_squared,28,100.000000
egfr_non_african_american_range_max_milliliter_per_minute_per_1.73_meter_squared,28,100.000000
bun_to_creatinine_ratio_value,22,78.571429
total_bilirubin_value_milligram_per_deciliter,1,3.571429
bun_to_creatinine_ratio_range_max,1,3.571429
bun_to_creatinine_ratio_range_min,1,3.571429



TOTAL MISSING CELLS: 81

TOTAL DATA CELLS: 1624


In [21]:
# ============================================================
# 08. SIMULATED ASTRONAUT MONITORING DATA
# ============================================================

print("ASTRA VITALIS — MONITORING DATA GENERATION")
print("=" * 50)

rng = np.random.default_rng(42)

days = 120

monitoring = pd.DataFrame({
    "Mission Day": np.arange(1, days + 1),
    "Heart Rate": rng.normal(72, 5, days),
    "Sleep Hours": rng.normal(7.5, 0.7, days),
    "Activity Minutes": rng.normal(65, 12, days),
    "Stress Score": rng.normal(3.5, 1.2, days),
    "SpO2": rng.normal(97, 1, days)
})

# Introduce selected unusual simulated records
monitoring.loc[15, "Heart Rate"] = 112
monitoring.loc[15, "Sleep Hours"] = 4.1

monitoring.loc[42, "Stress Score"] = 8.5

monitoring.loc[76, "Activity Minutes"] = 15

monitoring.loc[101, "SpO2"] = 90

print("Simulated monitoring records:", len(monitoring))

display(monitoring.head(10))

print("\nMONITORING DATA STATISTICS")
display(monitoring.describe().T)

ASTRA VITALIS — MONITORING DATA GENERATION
Simulated monitoring records: 120


,Mission Day,Heart Rate,Sleep Hours,Activity Minutes,Stress Score,SpO2
0,1,73.523585,6.783552,54.477671,3.316973,97.223080
1,2,66.800079,7.625493,63.868849,3.960073,98.433215
2,3,75.752256,7.653998,43.907259,4.699789,97.091520
3,4,76.702824,8.451431,47.395457,2.229757,97.580777
4,5,62.244824,8.084578,90.550965,3.349989,96.943217
5,6,65.489102,7.749810,49.550929,5.277747,96.829592
6,7,72.639202,8.524312,51.838573,2.607694,96.220518
7,8,70.418787,6.667866,87.042962,2.513300,97.430301
8,9,71.915994,7.052174,99.860806,3.742767,96.148463
9,10,67.734780,6.851397,50.941200,4.513262,97.665585



MONITORING DATA STATISTICS


,count,mean,std,min,25%,50%,75%,max
Mission Day,120.0,60.500000,34.785054,1.000000,30.750000,60.500000,90.250000,120.000000
Heart Rate,120.0,72.067128,5.364599,62.244824,69.417224,71.989232,74.675102,112.000000
Sleep Hours,120.0,7.454060,0.766536,4.100000,6.929463,7.396646,7.828358,9.539704
Activity Minutes,120.0,65.372398,13.162924,15.000000,57.666648,66.347577,73.829579,99.860806
Stress Score,120.0,3.519482,1.299425,0.699661,2.678160,3.539468,4.267840,8.500000
SpO2,120.0,96.859833,1.197064,90.000000,96.203518,96.921609,97.562594,99.420415


In [22]:
# ============================================================
# 09. ISOLATION FOREST ANOMALY DETECTION
# ============================================================

features = [
    "Heart Rate",
    "Sleep Hours",
    "Activity Minutes",
    "Stress Score",
    "SpO2"
]

X = monitoring[features]

model = IsolationForest(
    n_estimators=150,
    contamination=0.12,
    random_state=42
)

monitoring["Anomaly Prediction"] = model.fit_predict(X)

monitoring["Anomaly Status"] = monitoring[
    "Anomaly Prediction"
].map({
    1: "Normal",
    -1: "Unusual"
})

print("ISOLATION FOREST MODEL RESULTS")
print("=" * 45)

print("Algorithm: Isolation Forest")
print("Estimators: 150")
print("Contamination: 0.12")
print("Input indicators:", len(features))

display(
    monitoring["Anomaly Status"].value_counts().to_frame(
        "Record Count"
    )
)

ISOLATION FOREST MODEL RESULTS
Algorithm: Isolation Forest
Estimators: 150
Contamination: 0.12
Input indicators: 5


,Record Count
Anomaly Status,
Normal,105
Unusual,15


In [23]:
# ============================================================
# 10. MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Ground-truth labels for deliberately injected records
monitoring["Expected Anomaly"] = 0

injected_indices = [15, 42, 76, 101]

monitoring.loc[injected_indices, "Expected Anomaly"] = 1

# Model predictions: -1 = unusual, 1 = normal
monitoring["Predicted Anomaly"] = (
    monitoring["Anomaly Prediction"].map({
        1: 0,
        -1: 1
    })
)

y_true = monitoring["Expected Anomaly"]
y_pred = monitoring["Predicted Anomaly"]

print("MODEL EVALUATION REPORT")
print("=" * 50)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Injected Anomaly"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nAccuracy:", round(accuracy_score(y_true, y_pred), 3))

print("\nDELIBERATELY INJECTED RECORDS")
display(
    monitoring.loc[
        injected_indices,
        [
            "Mission Day",
            "Heart Rate",
            "Sleep Hours",
            "Activity Minutes",
            "Stress Score",
            "SpO2",
            "Anomaly Status"
        ]
    ]
)

MODEL EVALUATION REPORT

Classification Report:
                  precision    recall  f1-score   support

          Normal       1.00      0.91      0.95       116
Injected Anomaly       0.27      1.00      0.42         4

        accuracy                           0.91       120
       macro avg       0.63      0.95      0.69       120
    weighted avg       0.98      0.91      0.93       120

Confusion Matrix:
[[105  11]
 [  0   4]]

Accuracy: 0.908

DELIBERATELY INJECTED RECORDS


,Mission Day,Heart Rate,Sleep Hours,Activity Minutes,Stress Score,SpO2,Anomaly Status
15,16,112.000000,4.100000,74.328051,6.293184,97.935547,Unusual
42,43,68.672451,8.989929,76.129262,8.500000,98.335784,Unusual
76,77,69.863737,7.950329,15.000000,2.859320,96.246472,Unusual
101,102,78.496141,7.669748,68.652401,2.499957,90.000000,Unusual


In [24]:
# ============================================================
# 11. ANOMALY SCORE VISUALIZATION
# ============================================================

monitoring["Anomaly Score"] = -model.decision_function(X)

fig = px.scatter(
    monitoring,
    x="Mission Day",
    y="Anomaly Score",
    color="Anomaly Status",
    symbol="Expected Anomaly",
    hover_data=features,
    title="ASTRA VITALIS — Isolation Forest Anomaly Scores"
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Mission Day",
    yaxis_title="Anomaly Score"
)

fig.show()

In [25]:
# ============================================================
# 12. MONITORING INDICATOR ANALYSIS
# ============================================================

print("ASTRA VITALIS — FEATURE ANALYSIS")
print("=" * 50)

normal_records = monitoring[
    monitoring["Anomaly Status"] == "Normal"
]

unusual_records = monitoring[
    monitoring["Anomaly Status"] == "Unusual"
]

feature_comparison = pd.DataFrame({
    "Normal Mean": normal_records[features].mean(),
    "Unusual Mean": unusual_records[features].mean(),
    "Overall Mean": monitoring[features].mean()
})

display(feature_comparison.round(2))

fig = px.bar(
    feature_comparison.reset_index().melt(
        id_vars="index",
        var_name="Record Group",
        value_name="Average Value"
    ),
    x="index",
    y="Average Value",
    color="Record Group",
    barmode="group",
    title="Monitoring Indicators — Normal vs Unusual Records",
    labels={"index": "Monitoring Indicator"}
)

fig.update_layout(template="plotly_dark")
fig.show()

ASTRA VITALIS — FEATURE ANALYSIS


,Normal Mean,Unusual Mean,Overall Mean
Heart Rate,71.92,73.07,72.07
Sleep Hours,7.41,7.75,7.45
Activity Minutes,65.11,67.23,65.37
Stress Score,3.55,3.32,3.52
SpO2,96.92,96.45,96.86


In [26]:
# ============================================================
# 13. MISSION MONITORING TIMELINE
# ============================================================

fig = px.line(
    monitoring,
    x="Mission Day",
    y=["Heart Rate", "Sleep Hours", "Activity Minutes",
       "Stress Score", "SpO2"],
    title="ASTRA VITALIS — Simulated Mission Monitoring Timeline"
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Mission Day",
    yaxis_title="Recorded Indicator Value",
    legend_title="Monitoring Indicator"
)

fig.show()

In [27]:
# ============================================================
# 14. EXPORT RESEARCH RESULTS
# ============================================================

OUTPUT_DIR = Path("research_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

reference_analysis.to_csv(
    OUTPUT_DIR / "blood_chemistry_analysis.csv",
    index=False
)

monitoring.to_csv(
    OUTPUT_DIR / "anomaly_detection_results.csv",
    index=False
)

feature_comparison.to_csv(
    OUTPUT_DIR / "feature_comparison.csv"
)

print("RESEARCH OUTPUTS EXPORTED")
print("=" * 40)

for file in OUTPUT_DIR.iterdir():
    print("Saved:", file.name)

RESEARCH OUTPUTS EXPORTED
Saved: anomaly_detection_results.csv
Saved: blood_chemistry_analysis.csv
Saved: feature_comparison.csv


In [28]:
# ============================================================
# 15. ASTRA VITALIS — FINAL RESEARCH SUMMARY
# ============================================================

from sklearn.metrics import precision_recall_fscore_support

# NASA dataset findings
status_counts = reference_analysis["Status"].value_counts()

# Model evaluation
precision, recall, f1, _ = precision_recall_fscore_support(
    monitoring["Expected Anomaly"],
    monitoring["Predicted Anomaly"],
    average="binary",
    zero_division=0
)

detected_injected = (
    monitoring.loc[injected_indices, "Predicted Anomaly"] == 1
).sum()

research_summary = {
    "NASA Dataset Samples": len(nasa),
    "NASA Participants": nasa["Participant"].nunique(),
    "NASA Timepoints": nasa["Timepoint"].nunique(),
    "Blood Chemistry Comparisons": len(reference_analysis),
    "Within Reference Range": status_counts.get("Within", 0),
    "Below Reference Range": status_counts.get("Below", 0),
    "Above Reference Range": status_counts.get("Above", 0),
    "Simulated Monitoring Records": len(monitoring),
    "Model Flagged Records": (monitoring["Predicted Anomaly"] == 1).sum(),
    "Injected Anomalies Detected": int(detected_injected),
    "Model Precision": round(precision, 3),
    "Model Recall": round(recall, 3),
    "Model F1 Score": round(f1, 3)
}

print("ASTRA VITALIS — FINAL RESEARCH SUMMARY")
print("=" * 55)

display(
    pd.DataFrame.from_dict(
        research_summary,
        orient="index",
        columns=["Result"]
    )
)

print("\nRESEARCH SUMMARY GENERATED SUCCESSFULLY.")

ASTRA VITALIS — FINAL RESEARCH SUMMARY


,Result
NASA Dataset Samples,28.000
NASA Participants,4.000
NASA Timepoints,7.000
Blood Chemistry Comparisons,112.000
Within Reference Range,104.000
Below Reference Range,5.000
Above Reference Range,3.000
Simulated Monitoring Records,120.000
Model Flagged Records,15.000
Injected Anomalies Detected,4.000



RESEARCH SUMMARY GENERATED SUCCESSFULLY.


In [29]:
# incorrect
nasa.head()

# correct for NASA dataset analysis
nasa.head()

# or, if the code is about simulated monitoring:
monitoring.head()


,Mission Day,Heart Rate,Sleep Hours,Activity Minutes,Stress Score,SpO2,Anomaly Prediction,Anomaly Status,Expected Anomaly,Predicted Anomaly,Anomaly Score
0,1,73.523585,6.783552,54.477671,3.316973,97.223080,1,Normal,0,0,-0.106392
1,2,66.800079,7.625493,63.868849,3.960073,98.433215,1,Normal,0,0,-0.073484
2,3,75.752256,7.653998,43.907259,4.699789,97.091520,1,Normal,0,0,-0.061122
3,4,76.702824,8.451431,47.395457,2.229757,97.580777,1,Normal,0,0,-0.009223
4,5,62.244824,8.084578,90.550965,3.349989,96.943217,-1,Unusual,0,1,0.019973


# 16. Research Conclusion

# Research Overview
ASTRA_VITALIS_OVERVIEW = """
ASTRA VITALIS explores astronaut health monitoring through two complementary research tracks.
"""

### 1. NASA Inspiration4 Blood Chemistry Analysis

OSD-575 dataset was examined across four participants and seven mission-related timepoints.

Four blood chemistry parameters were investigated:
- Glucose
- Albumin
- Creatinine
- Sodium

Of 112 comparisons, 104 were within range, five were below range, and three were above range. These deviations are observations, not medical diagnoses.

### 2. Machine Learning Experimentation

An Isolation Forest model was applied to 120 simulated monitoring records using heart rate, sleep hours, activity minutes, stress score, and SpO2. It flagged 15 records, including all four deliberately injected anomalies. Because the data and injected anomalies are simulated, these results do not establish clinical performance.

### Research Limitations

- The NASA dataset contains only 28 samples from four participants.
- Reference-range deviations are not medical diagnoses.
- The monitoring dataset is simulated.
- The anomaly detection model has not been clinically validated.
- Artificially injected anomalies do not represent confirmed astronaut health events.
- The NASA blood chemistry dataset was analysed separately and was not used to train the Isolation Forest model.

### Future Research

Future work could investigate:

- Larger astronaut health datasets.
- Multivariate time-series analysis.
- More robust anomaly detection methods.
- Personalised monitoring baselines.
- Integration of validated physiological data sources.

### Final Statement

ASTRA VITALIS demonstrates a research prototype combining astronaut health data exploration, visual analytics, and machine-learning-based anomaly detection to investigate potential approaches to monitoring crew wellbeing during space missions.
- Albumin
- Creatinine
- Sodium

Their observed values were compared against the reference intervals supplied with the dataset.
- Albumin
- Creatinine
- Sodium

Their observed values were compared against the reference intervals supplied with the dataset.

### 2. Machine Learning Experimentation

An Isolation Forest model was trained on simulated astronaut monitoring records using:
- Heart Rate
- Sleep Hours
- Activity Minutes
- Stress Score
- SpO2

Controlled unusual observations were introduced to investigate the model's anomaly-detection behaviour.

### Research Limitations

- The NASA dataset contains only 28 samples from four participants.
- Reference-range deviations are not medical diagnoses.
- The monitoring dataset is simulated.
- The anomaly detection model has not been clinically validated.
- Artificially injected anomalies do not represent confirmed astronaut health events.
- The NASA blood chemistry dataset was analysed separately and was not used to train the Isolation Forest model.

### Future Research

Future work could investigate:
- Larger astronaut health datasets.
- Multivariate time-series analysis.
- More robust anomaly detection methods.
- Personalised monitoring baselines.
- Integration of validated physiological data sources.

### Final Statement

ASTRA VITALIS demonstrates a research prototype combining astronaut health data exploration, visual analytics, and machine-learning-based anomaly detection to investigate potential approaches to monitoring crew wellbeing during space missions.
